# TfR1 apical-domain binder — Proteina-Complexa on Colab

Challenge 04, Anthropic x Adaptyv. Target is transferrin receptor 1: bind human
TfR1 at pH 7.4, release at pH 6.0, cross-react with mouse.

Targeting the apical domain (UniProt P02786, residues 189-383). It's the domain
New World arenaviruses and parvovirus B19 use for entry, and where brain-shuttle
antibodies bind. Leaves the transferrin site on the helical domain free so the
receptor keeps recycling.



## Target prep

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib
CACHE  = pathlib.Path("/content/drive/MyDrive/tfr1_complexa")
REPO   = pathlib.Path("/content/Proteina-Complexa")
TGTDIR = CACHE / "PFM_data/target_data/tfr1"
WORK   = CACHE / "work"
for p in (TGTDIR, WORK, CACHE / "checkpoints", CACHE / "community_models/ckpts/AF2"):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q biopython

In [ ]:
import requests, json, time

# RCSB and UniProt both throw transient 500s. Got one mid-run and it poisoned the
# conservation map, so everything retries now.
def get(url, tries=4):
    for k in range(tries):
        try:
            r = requests.get(url, timeout=60)
            if r.ok:
                return r
        except Exception:
            pass
        time.sleep(2 * (k + 1))
    return None

def fetch_pdb(code):
    for c in (code.upper(), code.lower()):
        r = get("https://files.rcsb.org/download/%s.pdb" % c)
        if r is not None and "ATOM  " in r.text:
            return r.text
    return None

PDB_CODE = "11QC"
raw = fetch_pdb(PDB_CODE)
(TGTDIR / (PDB_CODE + ".pdb")).write_text(raw)

# 11RN = apical domain with parvovirus B19 VP1u bound. Using it to get a real
# epitope instead of guessing from geometry.
vp1u = fetch_pdb("11RN")
if vp1u:
    (TGTDIR / "11RN.pdb").write_text(vp1u)

def uniprot(acc):
    r = get("https://rest.uniprot.org/uniprotkb/%s.fasta" % acc)
    lines = r.text.strip().splitlines()
    return lines[0], "".join(lines[1:])

h_hdr, HUMAN = uniprot("P02786")   # TFR1_HUMAN
m_hdr, MOUSE = uniprot("Q62351")   # TFR1_MOUSE
print(h_hdr)
print(m_hdr)
(WORK / "sequences.json").write_text(json.dumps({"human": HUMAN, "mouse": MOUSE}))

In [ ]:
import warnings, numpy as np
warnings.filterwarnings("ignore")
from Bio.PDB import PDBParser
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Data.IUPACData import protein_letters_3to1

THREE2ONE = {k.upper(): v for k, v in protein_letters_3to1.items()}
parser = PDBParser(QUIET=True)
struct = parser.get_structure("tfr1", TGTDIR / (PDB_CODE + ".pdb"))[0]

def chain_seq(chain):
    ids, seq = [], []
    for res in chain:
        if res.id[0] == " " and res.get_resname() in THREE2ONE:
            ids.append(res.id[1])
            seq.append(THREE2ONE[res.get_resname()])
    return ids, "".join(seq)

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -11
aligner.extend_gap_score = -1

# Don't assume the PDB is numbered in UniProt coordinates. Deriving the offset.
# (11QC turns out to be offset 0, but checking anyway.)
offsets = {}
for ch in struct:
    ids, seq = chain_seq(ch)
    if len(seq) < 100:
        continue
    aln = aligner.align(HUMAN, seq)[0]
    offsets[ch.id] = (aln.aligned[0][0][0] + 1) - ids[aln.aligned[1][0][0]]
    print("chain", ch.id, len(seq), "residues, offset", offsets[ch.id])

In [ ]:
# Human vs mouse, per position, in human numbering.
aln_hm = aligner.align(HUMAN, MOUSE)[0]
identical = {}
for (h0, h1), (m0, m1) in zip(aln_hm.aligned[0], aln_hm.aligned[1]):
    for k in range(h1 - h0):
        identical[h0 + k + 1] = (HUMAN[h0 + k] == MOUSE[m0 + k])

APICAL = list(range(189, 384))
pct = 100.0 * np.mean([identical.get(i, False) for i in range(121, 761)])
print("ectodomain identity: %.1f%%" % pct)   # challenge says ~78%, matches
print("apical identical: %d/%d" % (sum(identical.get(i, False) for i in APICAL), len(APICAL)))

In [ ]:
from Bio.PDB.SASA import ShrakeRupley

CHAIN = "A"
off = offsets[CHAIN]
chA = struct[CHAIN]

sr = ShrakeRupley()
sr.compute(struct, level="R")
sasa = {r.id[1] + off: float(r.sasa) for r in chA if r.id[0] == " "}

# VP1u contacts on the receptor side.
# First pass here classified the 62-aa viral peptide as a TfR1 copy (short seq
# aligned against a 760-aa protein picks up enough spurious matches to clear a
# naive identity cutoff). That pulled in the receptor's own dimer interface at
# 637-756 as a fake "epitope". Requiring length >= 150 AND identity >= 0.8 fixes it.
vp1u_contacts = set()
if (TGTDIR / "11RN.pdb").exists():
    s2 = parser.get_structure("vp1u", TGTDIR / "11RN.pdb")[0]
    chains = [c for c in s2 if len(chain_seq(c)[1]) > 0]

    def ident_to_human(c):
        s = chain_seq(c)[1]
        a = aligner.align(HUMAN, s)[0]
        same = sum(1 for (h0, h1), (s0, s1) in zip(a.aligned[0], a.aligned[1])
                   for k in range(h1 - h0) if HUMAN[h0 + k] == s[s0 + k])
        return same / float(len(s))

    lens  = {c.id: len(chain_seq(c)[1]) for c in chains}
    ident = {c.id: ident_to_human(c) for c in chains}
    print("chains (len, identity):", {k: (lens[k], round(ident[k], 2)) for k in ident})

    is_tfr  = lambda c: lens[c.id] >= 150 and ident[c.id] >= 0.8
    tfr_ch  = max([c for c in chains if is_tfr(c)], key=lambda c: lens[c.id])
    partner = np.array([a.coord for c in chains if not is_tfr(c) for a in c.get_atoms()])

    ids_t, seq_t = chain_seq(tfr_ch)
    a2 = aligner.align(HUMAN, seq_t)[0]
    off_rn = (a2.aligned[0][0][0] + 1) - ids_t[a2.aligned[1][0][0]]

    for res in tfr_ch:
        if res.id[0] != " ":
            continue
        ra = np.array([a.coord for a in res])
        if np.linalg.norm(ra[:, None, :] - partner[None, :, :], axis=-1).min() < 5.0:
            vp1u_contacts.add(int(res.id[1] + off_rn))

    vp1u_contacts = {r for r in vp1u_contacts if 189 <= r <= 383}
    print("viral epitope:", sorted(vp1u_contacts))

In [ ]:
EXPOSED = 40.0
pool = [i for i in APICAL if identical.get(i, False) and sasa.get(i, 0.0) > EXPOSED]

# This is the finding. The epitope the challenge recommends is mostly
# species-divergent, so objective 1 (hit the functional site) and objective 2
# (mouse cross-reactivity) are in direct tension. Picking conserved patches
# over the viral site.
vc = sorted(vp1u_contacts)
print("viral epitope residues:", len(vc))
print("  mouse-identical:", [i for i in vc if identical.get(i, False)])
print("  also exposed:   ", [i for i in vc if i in pool])
print("conserved + exposed across whole apical domain:", len(pool))

In [ ]:
ca = {r.id[1] + off: r["CA"].coord for r in chA if r.id[0] == " " and "CA" in r}

def cluster(seed, k=4, cutoff=14.0):
    near = sorted([i for i in pool if i != seed and i in ca
                   and float(np.linalg.norm(ca[i] - ca[seed])) < cutoff],
                  key=lambda i: float(np.linalg.norm(ca[i] - ca[seed])))
    return sorted([seed] + near[:k - 1])

# Seed on conserved viral-epitope residues first, then most-exposed conserved
# residues elsewhere. Only 1 residue qualifies on the viral side, so without the
# fallback seeds I'd get a single epitope option.
seeds = [i for i in pool if i in vp1u_contacts]
seeds += [i for i in sorted(pool, key=lambda x: -sasa.get(x, 0.0)) if i not in seeds]

HOTSPOT_SETS, used = [], set()
for s in seeds:
    if s not in ca:
        continue
    hs = cluster(s)
    if len(hs) < 3 or hs in HOTSPOT_SETS or (used and len(set(hs) & used) > 1):
        continue
    HOTSPOT_SETS.append(hs)
    used |= set(hs)
    if len(HOTSPOT_SETS) == 3:
        break

for i, hs in enumerate(HOTSPOT_SETS):
    print("set %d: %s  mouse-identical=%s  viral-epitope members=%d"
          % (i, hs, all(identical.get(r, False) for r in hs),
             sum(1 for r in hs if r in vp1u_contacts)))

In [ ]:
import csv
with open(WORK / "conservation_apical.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["uniprot_pos", "human_aa", "identical_to_mouse", "sasa", "in_vp1u_epitope"])
    for i in APICAL:
        w.writerow([i, HUMAN[i - 1], identical.get(i, False),
                    round(sasa.get(i, 0.0), 1), i in vp1u_contacts])

In [ ]:
LO, HI = 189, 383

# Editing the raw PDB text rather than mutating Bio.PDB residue ids -- renumbering
# in place hits child_dict key collisions on save.
def crop_renumber(raw_text, chain_offsets, lo, hi):
    out = []
    for line in raw_text.splitlines():
        if not line.startswith("ATOM  "):
            continue
        if line[16] not in (" ", "A"):          # altloc
            continue
        ch = line[21]
        if ch not in chain_offsets:
            continue
        num = int(line[22:26]) + chain_offsets[ch]
        if not (lo <= num <= hi):
            continue
        if len(line) >= 78 and line[76:78].strip() == "H":
            continue
        out.append(line[:16] + " " + line[17:22] + ("%4d" % num) + line[26:].rstrip())
    return "\n".join(out) + "\nTER\nEND\n"

raw_text = (TGTDIR / (PDB_CODE + ".pdb")).read_text()
(TGTDIR / "tfr1_apical_A.pdb").write_text(
    crop_renumber(raw_text, {CHAIN: offsets[CHAIN]}, LO, HI))
if len(offsets) > 1:
    (TGTDIR / "tfr1_apical_AB.pdb").write_text(crop_renumber(raw_text, offsets, LO, HI))

# Cropping to the apical domain because the full ectodomain dimer is ~1280 residues
# and AF2-Multimer won't fit alongside the generator on one card. Defensible anyway:
# the apical domain has been expressed and crystallised standalone (6Y76, 8P0Z).
check = parser.get_structure("c", TGTDIR / "tfr1_apical_A.pdb")[0]["A"]
nums = [r.id[1] for r in check if r.id[0] == " "]
bad = [r.id[1] for r in check if r.id[0] == " "
       and HUMAN[r.id[1] - 1] != THREE2ONE.get(r.get_resname(), "?")]
print(len(nums), "residues,", min(nums), "-", max(nums), "| seq mismatches vs P02786:", len(bad))

In [ ]:
manifest = {
    "pdb_code": PDB_CODE,
    "chain": CHAIN,
    "offsets": {str(k): int(v) for k, v in offsets.items()},
    "apical_range": [LO, HI],
    "hotspot_sets": [[int(r) for r in hs] for hs in HOTSPOT_SETS],
    "human_mouse_identity_ectodomain_pct": round(float(pct), 1),
    "vp1u_contacts": sorted(int(r) for r in vp1u_contacts),
}
(WORK / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## Environment

Install notes, in the order I hit them:

1. `uv venv --python 3.12` is fine — uv fetches its own 3.12, so Colab's 3.13 isn't
   the problem I first thought it was.
2. **`UV_SYSTEM_PYTHON=1`.** Colab sets this so `uv pip install` works globally in
   notebooks. It overrides the activated venv, so every install in the build script
   went to `/usr` and `.venv` stayed empty. Step 3 printed
   `Using Python 3.13.16 environment at: /usr` and I scrolled straight past it.
   This one cost me most of the night.
3. `UV_HTTP_TIMEOUT` defaults to 120s vs a 2.5 GB torch wheel. Script has `set -e`,
   so it died silently — and I was piping it through `tail`.
4. `torchaudio` unpinned resolves to 2.11.0 against torch 2.7.0, ABI mismatch,
   `undefined symbol: torch_library_impl`. Their open issue #42.
5. `atomworks` unpinned gives 3.0.4, which dropped `AtomSelectionStack`. Needs <3.
6. `dm-haiku` too new for the pinned jax 0.4.29 — `jax.core.take_current_trace`.
   Downgrading haiku is the fix, not upgrading jax.

In [ ]:
!git clone --depth 1 https://github.com/NVIDIA-BioNeMo/Proteina-Complexa.git /content/Proteina-Complexa

In [ ]:
import subprocess, re, os

script = REPO / "env/build_uv_env.sh"
s = script.read_text()

s = s.replace("uv pip install torch==2.7.0+cu126 torchvision torchaudio \\",
              "uv pip install torch==2.7.0+cu126 torchvision torchaudio==2.7.0 \\", 1)

tmol = 'uv pip install "git+https://github.com/uw-ipd/tmol.git'
if "llvmlite" not in s:
    s = s.replace(tmol, 'uv pip install "llvmlite>=0.41" "numba>=0.59" || true\n    ' + tmol, 1)

script.write_text(s)
script.chmod(0o755)

env = dict(os.environ, UV_SYSTEM_PYTHON="0", UV_VENV_CLEAR="1",
           UV_HTTP_TIMEOUT="600", UV_CONCURRENT_DOWNLOADS="4")
env.pop("UV_PYTHON", None)

# Full log this time. Not piping through tail again.
with open("/content/build.log", "w") as f:
    p = subprocess.run(["bash", str(script)], cwd=str(REPO),
                       stdout=f, stderr=subprocess.STDOUT, env=env)

log = open("/content/build.log").read()
print("exit:", p.returncode)
for line in log.splitlines():
    if re.match(r"^\s*\[\d/\d\]", line) or "environment at:" in line:
        print(" ", line.strip())

In [ ]:
# Step 3 above must say the venv, not /usr. That one line is the whole bug.
VPY = REPO / ".venv/bin/python"

def vpy(label, code):
    r = subprocess.run([str(VPY), "-c", code], capture_output=True, text=True)
    print("%-22s %s" % (label, (r.stdout + r.stderr).strip()[:110]))

vpy("python",       "import sys; print(sys.version.split()[0])")
vpy("torch",        "import torch; print(torch.__version__, torch.cuda.is_available())")
vpy("torchaudio",   "import torchaudio; print(torchaudio.__version__)")
vpy("proteinfound", "import proteinfoundation; print('ok')")
vpy("atomworks",    "import atomworks; print(atomworks.__version__)")
vpy("graphein",     "import graphein; print('ok')")
vpy("torch_scatter","from torch_scatter import scatter_mean; print('ok')")
vpy("colabdesign",  "import colabdesign; print('ok')")

In [ ]:
# Post-install pins. The build script leaves the bio stack floating, so anything
# released since the repo went up drifts.
# NOTE: --python is required. Without it uv silently targets /usr again and the
# pins look like they applied but didn't.
env_pin = {k: v for k, v in os.environ.items() if k != "UV_SYSTEM_PYTHON"}
env_pin["UV_SYSTEM_PYTHON"] = "0"

for pkg in ["atomworks[ml]==2.2.1", "dm-haiku==0.0.13"]:
    r = subprocess.run(["uv", "pip", "install", "--python", str(VPY), "-q", pkg],
                       capture_output=True, text=True, env=env_pin, cwd=str(REPO))
    print(pkg, "rc", r.returncode)

vpy("atomworks",   "import atomworks; print(atomworks.__version__)")
vpy("AtomSelStack","from atomworks.io.utils.selection import AtomSelectionStack; print('ok')")
vpy("colabdesign", "import colabdesign; print('ok')")

## Weights

In [ ]:
# Straight from HF by URL. complexa download wants NGC and stalls on the
# interactive wizard in a notebook. -c resumes, and these live on Drive so a
# recycled VM doesn't cost another 7 GB.
BASE = ("https://huggingface.co/nvidia/"
        "NV-Proteina-Complexa-Protein-Target-160M-v1/resolve/main")
CKPT = CACHE / "checkpoints"

for name, floor in [("complexa.ckpt", 2.5e9), ("complexa_ae.ckpt", 3.5e9)]:
    d = CKPT / name
    if d.exists() and d.stat().st_size >= floor:
        print(name, "%.2f GB" % (d.stat().st_size / 1e9))
        continue
    subprocess.run(["wget", "-c", "-q", "--show-progress",
                    "%s/%s" % (BASE, name), "-O", str(d)])

for name in ("complexa.ckpt", "complexa_ae.ckpt"):
    print(name, "%.2f GB" % ((CKPT / name).stat().st_size / 1e9))

In [ ]:
AF2 = CACHE / "community_models/ckpts/AF2"
(AF2 / "params").mkdir(parents=True, exist_ok=True)
if not list((AF2 / "params").glob("params_model_*multimer*.npz")):
    subprocess.run(["wget", "-q", "--show-progress",
                    "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar",
                    "-O", "/tmp/af2.tar"])
    subprocess.run(["tar", "-xf", "/tmp/af2.tar", "-C", str(AF2 / "params")])
    os.remove("/tmp/af2.tar")
print(len(list((AF2 / "params").glob("params_model_*multimer*.npz"))), "multimer param files")

## Config

In [ ]:
import yaml, shlex

(REPO / ".env").write_text("\n".join([
    "AF2_DIR=" + str(AF2),
    "ESM_DIR=" + str(CACHE / "community_models/ckpts/ESM2"),
    "SC_EXEC=" + str(CACHE / "tools/sc"),
    "DSSP_EXEC=" + str(CACHE / "tools/dssp"),
    "FOLDSEEK_EXEC=/usr/bin/foldseek",
    "MMSEQS_EXEC=/usr/bin/mmseqs",
    "LOCAL_CODE_PATH=" + str(REPO),
    "COMMUNITY_MODELS_PATH=" + str(REPO / "community_models"),
    "DATA_PATH=" + str(CACHE / "PFM_data"),
    "",
]))

# Appending, not rewriting -- safe_dump would drop the repo's comments.
TARGETS = REPO / "configs/targets/targets_dict.yaml"
existing = TARGETS.read_text()
if "tfr1_apical_h0" not in existing:
    block = []
    for i, hs in enumerate(HOTSPOT_SETS):
        block += ["  tfr1_apical_h%d:" % i,
                  "    source: tfr1",
                  "    target_filename: tfr1_apical_A",
                  "    target_input: A189-383",
                  "    hotspot_residues: [%s]" % ", ".join("A%d" % r for r in hs),
                  "    binder_length: [55, 95]",
                  "    pdb_id: %s" % PDB_CODE.lower()]
    TARGETS.write_text(existing.rstrip("\n") + "\n" + "\n".join(block) + "\n")

cfg = yaml.safe_load(TARGETS.read_text())["target_dict_cfg"]
print(yaml.safe_dump({k: v for k, v in cfg.items() if k.startswith("tfr1")},
                     sort_keys=False, default_flow_style=False))

(WORK / "ckpts.env").write_text(
    "export CKPT_PATH=" + shlex.quote(str(CKPT)) + "\n"
    "export CKPT_NAME=complexa.ckpt\n"
    "export AE_PATH=" + shlex.quote(str(CKPT / "complexa_ae.ckpt")) + "\n")

In [ ]:
%%bash
# The CLI gates on COMPLEXA_INIT, which only env.sh exports -- writing .env by hand
# isn't enough.
cd /content/Proteina-Complexa
source .venv/bin/activate
complexa init uv --force
source env.sh
echo "COMPLEXA_INIT=$COMPLEXA_INIT"

## Generate

In [ ]:
%%bash
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh
source /content/drive/MyDrive/tfr1_complexa/work/ckpts.env

# --verbose between the config positional and the overrides breaks argparse's
# positional group ("unrecognized arguments"). Leaving it off.
#
# JAX preallocates ~75% of VRAM for the AF2 reward before PyTorch gets any --
# 11 GB held on a 15 GB T4, and generation OOMs needing 284 MiB more.
export XLA_PYTHON_CLIENT_PREALLOCATE=false
export XLA_PYTHON_CLIENT_MEM_FRACTION=0.25
export XLA_PYTHON_CLIENT_ALLOCATOR=platform
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

complexa design configs/search_binder_local_pipeline.yaml \
  ++ckpt_path="$CKPT_PATH" ++ckpt_name="$CKPT_NAME" ++autoencoder_ckpt_path="$AE_PATH" \
  ++run_name=tfr1_smoke \
  ++generation.task_name=tfr1_apical_h0 \
  ++generation.args.nsteps=100 \
  ++generation.dataloader.dataset.nres.nsamples=2 \
  ++generation.dataloader.batch_size=1 \
  ++gen_njobs=1 ++eval_njobs=1 2>&1 | tail -30

In [ ]:
%%bash
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh
source /content/drive/MyDrive/tfr1_complexa/work/ckpts.env

export XLA_PYTHON_CLIENT_PREALLOCATE=false
export XLA_PYTHON_CLIENT_MEM_FRACTION=0.25
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

complexa design configs/search_binder_local_pipeline.yaml \
  ++ckpt_path="$CKPT_PATH" ++ckpt_name="$CKPT_NAME" ++autoencoder_ckpt_path="$AE_PATH" \
  ++run_name=tfr1_bon_h0 \
  ++generation.task_name=tfr1_apical_h0 \
  ++generation.args.nsteps=200 \
  ++generation.dataloader.dataset.nres.nsamples=64 \
  ++generation.filter.filter_samples_limit=24 \
  ++gen_njobs=1 ++eval_njobs=1 2>&1 | tail -30

In [ ]:
# Stage logs go in a directory, not a flat file -- logs/design_pipeline_<target>_<run>_<ts>/generate.log
import glob
d = sorted(glob.glob("/content/Proteina-Complexa/logs/design_pipeline_tfr1_*/"))[-1]
print(d)
print(open(d + "generate.log").read()[-3000:])

In [ ]:
# Searching broadly -- output location depends on config, so not assuming a dir name.
import shutil, time
dest = WORK / "raw_outputs"
dest.mkdir(parents=True, exist_ok=True)
CUTOFF = time.time() - 6 * 3600
SKIP = {".git", "checkpoints", ".venv", ".uv-cache", "community_models"}
KNOWN = {"11QC.pdb", "11RN.pdb", "tfr1_apical_A.pdb", "tfr1_apical_AB.pdb"}

found = []
for root in ("/content/Proteina-Complexa", str(CACHE / "PFM_data")):
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [x for x in dirnames if x not in SKIP]
        for fn in filenames:
            if fn.endswith((".pdb", ".csv", ".fa", ".fasta")) and fn not in KNOWN:
                p = os.path.join(dirpath, fn)
                if os.stat(p).st_mtime > CUTOFF:
                    found.append(p)

for p in found:
    shutil.copy(p, dest / os.path.relpath(p, "/content").replace(os.sep, "__"))
print("copied", len(found))

---

Ran out of GPU here. Colab moved me off the A100 onto a T4 partway through, and I
never got to test whether the JAX preallocation flags actually clear the OOM.

Generated the submitted designs through NVIDIA's hosted RFD4-Proteina endpoint
instead — see the other notebook. Same target crop and hotspots, no local GPU.

Still to do: AF2/Boltz refolding to filter on interface pAE and pLDDT, and the pH
work (histidine substitution at interface positions, imidazole pKa ~6.0-6.5 so
neutral at 7.4 and protonated at 6.0).